# 08 · Final analysis

Merge every result into master tables, run the Friedman and Wilcoxon tests, and draw the paper tables and figures.

- **Reads:** results of notebooks 03 to 07 and the DL baseline CSVs
- **Writes:** `master_*.csv`, `Table_1..9`, figures, statistical report

Written for Kaggle (`/kaggle/input/...`, `/kaggle/working/...`; the Kaggle account name is replaced by `<kaggle-user>`). Outputs are kept as saved. Read `docs/PIPELINE.md` for the order and for what is missing.

In [1]:
import os
import sys
from pathlib import Path

print("="*60)
print("🚀 NB08: BULLETPROOF FILE DISCOVERY & VALIDATION")
print("="*60)

KAGGLE_INPUT = Path('/kaggle/input')

def find_dir_containing(filename):
    """Searches all of /kaggle/input and returns the directory containing the file."""
    for root, dirs, files in os.walk(KAGGLE_INPUT):
        if filename in files:
            return Path(root)
    return None

print("\n🔍 Locating Dataset Roots via Unique Anchor Files...")

# Define a globally unique anchor file for each required notebook
ANCHORS = {
    'NASA': 'readme.txt',
    'NB1': 'dataset_manifest.json',
    'NB2': 'feature_metadata.json',
    'NB3': 'centralized_results.csv',
    'NB4': 'distributed_results.csv',
    'NB5': 'robustness_results.csv',
    'NB6': 'cross_dataset_results.csv',
    'NB7': 'rul_results.csv',
    'DL':  'centralized_dl_results.csv'  # <--- FIXED ANCHOR
}

DATASET_DIRS = {}
missing_anchors = []

for nb_name, anchor_file in ANCHORS.items():
    found_dir = find_dir_containing(anchor_file)
    if found_dir:
        DATASET_DIRS[nb_name] = found_dir
        print(f"   ✅ {nb_name} mapped to -> {found_dir.name}")
    else:
        missing_anchors.append(f"{nb_name} (Missing '{anchor_file}')")

if missing_anchors:
    print("\n❌ CRITICAL ERROR: Could not locate datasets for the following:")
    for ma in missing_anchors:
        print(f"   - {ma}")
    print("\nEnsure all datasets are attached and files are present.")
    sys.exit(1)

print("\n🔍 Validating remaining required artifacts (handling flattened Kaggle structure)...")

# Because Kaggle flattened the folders, we do not use 'results/' or 'metadata/' prefixes.
REQUIRED_FILES = {
    'NB1': ['dataset_manifest.csv', 'sensor_metadata.json', 'split_metadata.json'],
    'NB2': ['feature_manifest.csv', 'feature_columns.json'],
    'NB3': ['deployment_metrics.csv', 'threshold_summary.csv'],
    'NB4': ['deployment_metrics.csv'],
    'NB5': ['robustness_summary.csv', 'deployment_metrics.csv'],
    'NB6': ['deployment_metrics.csv', 'generalization_summary.csv'],
    'NB7': ['summary.csv', 'deployment_metrics.csv'],
    'DL':  ['dl_deployment_metrics.csv', 'dl_robustness_results.csv', 'dl_cross_dataset_results.csv']
}

missing_files = []
for nb_name, files in REQUIRED_FILES.items():
    base_dir = DATASET_DIRS[nb_name]
    for f in files:
        if not (base_dir / f).exists():
            missing_files.append(f"{nb_name} -> {f}")

if missing_files:
    print("\n❌ CRITICAL ERROR: Found the datasets, but specific files are missing:")
    for mf in missing_files:
        print(f"   [MISSING] {mf}")
    sys.exit(1)
    
print("✅ All required notebook artifacts are present and accounted for.")

# Expose the directories globally for the rest of the notebook to use
DIR_NB1 = DATASET_DIRS['NB1']
DIR_NB2 = DATASET_DIRS['NB2']
DIR_NB3 = DATASET_DIRS['NB3']
DIR_NB4 = DATASET_DIRS['NB4']
DIR_NB5 = DATASET_DIRS['NB5']
DIR_NB6 = DATASET_DIRS['NB6']
DIR_NB7 = DATASET_DIRS['NB7']
DIR_DL  = DATASET_DIRS['DL']
DIR_NASA = DATASET_DIRS['NASA']

# ---------------------------------------------------------
# 3. NB08 MASTER OUTPUT DIRECTORY CREATION
# ---------------------------------------------------------
ROOT_OUT = Path('/kaggle/working/NB-8-AGAIN-outputs')

DIRECTORIES = [
    ROOT_OUT / 'master_tables',
    ROOT_OUT / 'figures',
    ROOT_OUT / 'statistical_tests',
    ROOT_OUT / 'comparison',
    ROOT_OUT / 'metadata',
    ROOT_OUT / 'paper_tables'
]

print("\n📁 Initializing NB08 Output Architecture...")
for directory in DIRECTORIES:
    directory.mkdir(parents=True, exist_ok=True)
    print(f"   - Created: {directory.relative_to('/kaggle/working')}/")

print("\n" + "="*60)
print("🎯 VALIDATION PASSED. YOU ARE CLEARED TO COMMENCE PAPER GENERATION.")
print("="*60)

🚀 NB08: BULLETPROOF FILE DISCOVERY & VALIDATION

🔍 Locating Dataset Roots via Unique Anchor Files...
   ✅ NASA mapped to -> nasa-cmaps
   ✅ NB1 mapped to -> nb-1-again-outputs
   ✅ NB2 mapped to -> nb-3-again-outputs
   ✅ NB3 mapped to -> nb-03-again-outputs
   ✅ NB4 mapped to -> nb-4-again-outputs
   ✅ NB5 mapped to -> nb-5-again-outputs
   ✅ NB6 mapped to -> nb-6-again-outputs
   ✅ NB7 mapped to -> nb-7-again-outputs
   ✅ DL mapped to -> nb-dl-outputs

🔍 Validating remaining required artifacts (handling flattened Kaggle structure)...
✅ All required notebook artifacts are present and accounted for.

📁 Initializing NB08 Output Architecture...
   - Created: NB-8-AGAIN-outputs/master_tables/
   - Created: NB-8-AGAIN-outputs/figures/
   - Created: NB-8-AGAIN-outputs/statistical_tests/
   - Created: NB-8-AGAIN-outputs/comparison/
   - Created: NB-8-AGAIN-outputs/metadata/
   - Created: NB-8-AGAIN-outputs/paper_tables/

🎯 VALIDATION PASSED. YOU ARE CLEARED TO COMMENCE PAPER GENERATION.


In [2]:
import pandas as pd
import json
import matplotlib.pyplot as plt
import numpy as np

print("="*60)
print("📊 CELL 2: GENERATING MASTER CSVS & PUBLICATION TABLES 1 & 2")
print("="*60)

# ---------------------------------------------------------
# 1. MASTER CSV AGGREGATION
# ---------------------------------------------------------
print("\n📦 Building Master Databases...")

def safe_load(dir_path, filename):
    file_path = dir_path / filename
    if file_path.exists():
        return pd.read_csv(file_path)
    print(f"   [WARNING] {filename} missing in {dir_path.name}")
    return pd.DataFrame()

# 1a. Master Classification
df_cent = safe_load(DIR_NB3, 'centralized_results.csv').assign(Experiment_Phase='NB3_Centralized')
df_dist = safe_load(DIR_NB4, 'distributed_results.csv').assign(Experiment_Phase='NB4_Distributed')
master_class = pd.concat([df_cent, df_dist], ignore_index=True)
master_class.to_csv(ROOT_OUT / 'master_tables/master_classification_results.csv', index=False)

# 1b. Master Deployment
dep_nb3 = safe_load(DIR_NB3, 'deployment_metrics.csv').assign(Source='NB3_Centralized')
dep_nb4 = safe_load(DIR_NB4, 'deployment_metrics.csv').assign(Source='NB4_Distributed')
dep_nb5 = safe_load(DIR_NB5, 'deployment_metrics.csv').assign(Source='NB5_Robustness')
dep_nb6 = safe_load(DIR_NB6, 'deployment_metrics.csv').assign(Source='NB6_Transfer')
dep_nb7 = safe_load(DIR_NB7, 'deployment_metrics.csv').assign(Source='NB7_RUL')
dep_dl  = safe_load(DIR_DL,  'dl_deployment_metrics.csv').assign(Source='DL_Models')
master_dep = pd.concat([dep_nb3, dep_nb4, dep_nb5, dep_nb6, dep_nb7, dep_dl], ignore_index=True)
master_dep.to_csv(ROOT_OUT / 'master_tables/master_deployment_metrics.csv', index=False)

# 1c. Master Robustness
rob_ml = safe_load(DIR_NB5, 'robustness_results.csv').assign(Model_Family='ML')
rob_dl = safe_load(DIR_DL, 'dl_robustness_results.csv').assign(Model_Family='DL')
master_rob = pd.concat([rob_ml, rob_dl], ignore_index=True)
master_rob.to_csv(ROOT_OUT / 'master_tables/master_robustness_results.csv', index=False)

# 1d. Master Cross-Dataset
cross_ml = safe_load(DIR_NB6, 'cross_dataset_results.csv').assign(Model_Family='ML')
cross_dl = safe_load(DIR_DL, 'dl_cross_dataset_results.csv').assign(Model_Family='DL')
master_cross = pd.concat([cross_ml, cross_dl], ignore_index=True)
master_cross.to_csv(ROOT_OUT / 'master_tables/master_cross_dataset_results.csv', index=False)

# 1e. Master RUL & DL
master_rul = safe_load(DIR_NB7, 'rul_results.csv')
master_rul.to_csv(ROOT_OUT / 'master_tables/master_rul_results.csv', index=False)

master_dl = safe_load(DIR_DL, 'centralized_dl_results.csv')
master_dl.to_csv(ROOT_OUT / 'master_tables/master_dl_results.csv', index=False)

# 1f. Ultimate Master Database (Sparse consolidation of everything)
master_all = pd.concat([
    master_class, master_dep, master_rob, master_cross, master_rul, master_dl
], ignore_index=True)
master_all.to_csv(ROOT_OUT / 'master_tables/master_all_results.csv', index=False)

print("   ✅ Master CSVs successfully built and saved!")

# ---------------------------------------------------------
# 2. PNG RENDERER HELPER
# ---------------------------------------------------------
def render_table_to_png(df, filename, title):
    """Renders a pandas DataFrame as a publication-ready PNG."""
    fig, ax = plt.subplots(figsize=(max(8, len(df.columns)*1.8), len(df)*0.5 + 1.2))
    ax.axis('tight')
    ax.axis('off')
    
    table = ax.table(cellText=df.values, colLabels=df.columns, loc='center', cellLoc='center')
    table.auto_set_font_size(False)
    table.set_fontsize(11)
    table.scale(1.2, 1.8)
    
    # Format Headers and Rows
    for (row, col), cell in table.get_celld().items():
        if row == 0:
            cell.set_text_props(weight='bold', color='white')
            cell.set_facecolor('#2b3e50')  # Dark academic blue
        elif row % 2 == 0:
            cell.set_facecolor('#f4f4f4')  # Alternating row color
            
    plt.title(title, weight='bold', size=14, pad=20)
    plt.savefig(ROOT_OUT / f'paper_tables/{filename}.png', bbox_inches='tight', dpi=300)
    plt.close()

# ---------------------------------------------------------
# 3. TABLE 1: DATASET SUMMARY
# ---------------------------------------------------------
print("\n📝 Generating Table 1: Dataset Summary...")
table1_data = []
datasets = ['FD001', 'FD002', 'FD003', 'FD004']
ops_cond = {'FD001': 1, 'FD002': 6, 'FD003': 1, 'FD004': 6}
fault_modes = {'FD001': 1, 'FD002': 1, 'FD003': 2, 'FD004': 2}

# Safely extract from NB1 JSON if available, else fallback to CMAPSS constants
try:
    with open(DIR_NB1 / 'split_metadata.json', 'r') as f:
        split_meta = json.load(f)
    for ds in datasets:
        train_s = split_meta.get(ds, {}).get('train_samples', "N/A")
        test_s = split_meta.get(ds, {}).get('test_samples', "N/A")
        engines = split_meta.get(ds, {}).get('train_engines', "N/A")
        
        table1_data.append({
            'Dataset': ds, 'Engines (Train)': engines, 'Op Conditions': ops_cond[ds],
            'Fault Modes': fault_modes[ds], 'Train Samples': train_s, 'Test Samples': test_s
        })
except Exception:
    table1_data = [
        {'Dataset': 'FD001', 'Engines (Train)': 100, 'Op Conditions': 1, 'Fault Modes': 1, 'Train Samples': 20631, 'Test Samples': 13096},
        {'Dataset': 'FD002', 'Engines (Train)': 260, 'Op Conditions': 6, 'Fault Modes': 1, 'Train Samples': 53759, 'Test Samples': 33991},
        {'Dataset': 'FD003', 'Engines (Train)': 100, 'Op Conditions': 1, 'Fault Modes': 2, 'Train Samples': 24720, 'Test Samples': 16596},
        {'Dataset': 'FD004', 'Engines (Train)': 249, 'Op Conditions': 6, 'Fault Modes': 2, 'Train Samples': 61249, 'Test Samples': 41214}
    ]

table1_df = pd.DataFrame(table1_data)
table1_df.to_csv(ROOT_OUT / 'paper_tables/Table_1_Dataset_Summary.csv', index=False)
render_table_to_png(table1_df, 'Table_1_Dataset_Summary', 'Table 1: CMAPSS Dataset Summary')
print("   ✅ Table 1 rendered and saved.")

# ---------------------------------------------------------
# 4. TABLE 2: FEATURE SUMMARY
# ---------------------------------------------------------
print("\n📝 Generating Table 2: Feature Summary...")
try:
    with open(DIR_NB2 / 'feature_metadata.json', 'r') as f:
        feat_meta = json.load(f)
        
    active_s = ", ".join(map(str, feat_meta.get('active_sensors', [])))
    removed_s = ", ".join(map(str, feat_meta.get('removed_sensors', [])))
    
    table2_data = [
        {'Feature Category': 'Base Sensor/Setting Features', 'Quantity/Details': feat_meta.get('base_feature_count', 21)},
        {'Feature Category': 'Engineered Features (Lags, Roll, etc.)', 'Quantity/Details': feat_meta.get('engineered_feature_count', 'N/A')},
        {'Feature Category': 'Total Active Features', 'Quantity/Details': feat_meta.get('total_features', 'N/A')},
        {'Feature Category': 'Active Sensors Retained', 'Quantity/Details': active_s if active_s else "All dynamic sensors"},
        {'Feature Category': 'Removed Sensors (Flatlined)', 'Quantity/Details': removed_s if removed_s else "None"}
    ]
except Exception:
    table2_data = [{'Feature Category': 'Extraction Failed', 'Quantity/Details': 'Check feature_metadata.json'}]

table2_df = pd.DataFrame(table2_data)
table2_df.to_csv(ROOT_OUT / 'paper_tables/Table_2_Feature_Summary.csv', index=False)
render_table_to_png(table2_df, 'Table_2_Feature_Summary', 'Table 2: Feature Engineering Summary')
print("   ✅ Table 2 rendered and saved.")

print("\n" + "="*60)
print("🎯 MASTER CSVS AND TABLES 1 & 2 ARE COMPLETED.")
print("="*60)

📊 CELL 2: GENERATING MASTER CSVS & PUBLICATION TABLES 1 & 2

📦 Building Master Databases...
   ✅ Master CSVs successfully built and saved!

📝 Generating Table 1: Dataset Summary...
   ✅ Table 1 rendered and saved.

📝 Generating Table 2: Feature Summary...
   ✅ Table 2 rendered and saved.

🎯 MASTER CSVS AND TABLES 1 & 2 ARE COMPLETED.


In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

print("="*60)
print("📊 CELL 3: GENERATING TABLES 3 & 4 (CLASSICAL & DISTRIBUTED)")
print("="*60)

ROOT_OUT = Path('/kaggle/working/NB-8-AGAIN-outputs')
df_class = pd.read_csv(ROOT_OUT / 'master_tables/master_classification_results.csv')

# --- DYNAMIC COLUMN DISCOVERY ---
# This prevents KeyErrors if the columns are lowercase ('model', 'dataset', etc.)
model_col = next((c for c in df_class.columns if c.lower() in ['model', 'classifier', 'algorithm']), 'Model')
dataset_col = next((c for c in df_class.columns if c.lower() == 'dataset'), 'Dataset')

def format_mean_std(group, target_metric):
    """Formats Mean ± Std for a given metric dynamically."""
    # Find exact metric column name (case-insensitive)
    exact_col = next((c for c in group.columns if c.lower() == target_metric.lower()), None)
    if exact_col is None or group[exact_col].isnull().all():
        return "N/A"
    return f"{group[exact_col].mean():.3f} ± {group[exact_col].std():.3f}"

# ---------------------------------------------------------
# TABLE 3: Centralized ML Performance (Pivoted by Dataset)
# ---------------------------------------------------------
print("\n📝 Generating Table 3: Centralized Performance...")
df_cent = df_class[df_class['Experiment_Phase'] == 'NB3_Centralized']

if not df_cent.empty:
    # Calculate Mean ± Std for F1 and AUC
    t3_agg = df_cent.groupby([model_col, dataset_col]).apply(
        lambda x: pd.Series({
            'F1': format_mean_std(x, 'f1'),
            'AUC': format_mean_std(x, 'roc_auc')
        })
    ).reset_index()

    # Pivot to make it publication-wide: Rows = Models, Columns = Datasets
    t3_pivot = t3_agg.pivot(index=model_col, columns=dataset_col, values=['F1', 'AUC'])
    t3_pivot.columns = [f"{col[1]} ({col[0]})" for col in t3_pivot.columns] # Flatten MultiIndex
    t3_pivot.reset_index(inplace=True)
    t3_pivot.rename(columns={model_col: 'Model'}, inplace=True)
    
    t3_pivot.to_csv(ROOT_OUT / 'paper_tables/Table_3_Centralized_ML.csv', index=False)
    render_table_to_png(t3_pivot, 'Table_3_Centralized_ML', 'Table 3: Centralized ML Performance (Mean ± Std)')
    print("   ✅ Table 3 saved to paper_tables/")
else:
    print("   ❌ Centralized data missing.")

# ---------------------------------------------------------
# TABLE 4: Distributed ML Performance
# ---------------------------------------------------------
print("\n📝 Generating Table 4: Distributed Performance...")
df_dist = df_class[df_class['Experiment_Phase'] == 'NB4_Distributed']

if not df_dist.empty:
    # Identify the column containing the voting strategy
    vote_cols = [c for c in df_dist.columns if 'vote' in c.lower() or 'method' in c.lower() or 'strategy' in c.lower()]
    group_col = vote_cols[0] if vote_cols else model_col # Fallback

    t4_agg = df_dist.groupby([group_col, dataset_col]).apply(
        lambda x: pd.Series({
            'F1': format_mean_std(x, 'f1'),
            'AUC': format_mean_std(x, 'roc_auc')
        })
    ).reset_index()

    # Pivot: Rows = Voting Strategy, Columns = Datasets
    t4_pivot = t4_agg.pivot(index=group_col, columns=dataset_col, values=['F1', 'AUC'])
    t4_pivot.columns = [f"{col[1]} ({col[0]})" for col in t4_pivot.columns]
    t4_pivot.reset_index(inplace=True)
    t4_pivot.rename(columns={group_col: 'Aggregation Strategy'}, inplace=True)
    
    t4_pivot.to_csv(ROOT_OUT / 'paper_tables/Table_4_Distributed_ML.csv', index=False)
    render_table_to_png(t4_pivot, 'Table_4_Distributed_ML', 'Table 4: Distributed Aggregation Performance')
    print("   ✅ Table 4 saved to paper_tables/")
else:
    print("   ❌ Distributed data missing.")

📊 CELL 3: GENERATING TABLES 3 & 4 (CLASSICAL & DISTRIBUTED)

📝 Generating Table 3: Centralized Performance...


/tmp/ipykernel_58/2810872472.py:34: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  t3_agg = df_cent.groupby([model_col, dataset_col]).apply(


   ✅ Table 3 saved to paper_tables/

📝 Generating Table 4: Distributed Performance...


/tmp/ipykernel_58/2810872472.py:64: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  t4_agg = df_dist.groupby([group_col, dataset_col]).apply(


   ✅ Table 4 saved to paper_tables/


In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

print("="*60)
print("📊 CELL 4: GENERATING TABLES 5 & 6 (ROBUSTNESS & TRANSFER)")
print("="*60)

ROOT_OUT = Path('/kaggle/working/NB-8-AGAIN-outputs')

def get_col(df, keywords):
    """Safely finds a column name prioritizing exact matches, then partial matches."""
    # 1. Prioritize exact matches (prevents grabbing 'performance_drop_f1' instead of 'f1')
    for c in df.columns:
        if c.lower() in [k.lower() for k in keywords]:
            return c
    # 2. Fallback to partial matches
    for c in df.columns:
        if any(k.lower() in c.lower() for k in keywords):
            return c
    return None

# ---------------------------------------------------------
# TABLE 5: Robustness Degradation
# ---------------------------------------------------------
print("\n📝 Generating Table 5: Robustness Degradation...")
try:
    df_rob = pd.read_csv(ROOT_OUT / 'master_tables/master_robustness_results.csv')
    
    if not df_rob.empty:
        # Added 'perturbation_type' and 'perturbation_level' to the keywords
        pert_col = get_col(df_rob, ['perturbation_type', 'perturb', 'type', 'fault'])
        int_col = get_col(df_rob, ['perturbation_level', 'level', 'intensity', 'deg', 'noise'])
        f1_col = get_col(df_rob, ['f1', 'f-1'])
        
        if pert_col and int_col and f1_col:
            # Aggregate Mean F1 per Perturbation & Intensity
            t5_agg = df_rob.groupby([pert_col, int_col]).agg({f1_col: 'mean'}).reset_index()
            t5_agg.rename(columns={pert_col: 'Perturbation Type', int_col: 'Intensity (%)', f1_col: 'Mean F1 Score'}, inplace=True)
            t5_agg['Mean F1 Score'] = t5_agg['Mean F1 Score'].round(4)
            
            t5_agg.to_csv(ROOT_OUT / 'paper_tables/Table_5_Robustness.csv', index=False)
            render_table_to_png(t5_agg.head(15), 'Table_5_Robustness', 'Table 5: Model Robustness (Top 15 Rows)')
            print("   ✅ Table 5 saved to paper_tables/")
        else:
            print(f"   ❌ Missing columns in robustness CSV. Found: {df_rob.columns.tolist()}")
    else:
        print("   ❌ Robustness data is empty.")
except FileNotFoundError:
    print("   ❌ master_robustness_results.csv not found.")

# ---------------------------------------------------------
# TABLE 6: Cross-Dataset Transfer Matrix
# ---------------------------------------------------------
print("\n📝 Generating Table 6: Cross-Dataset Transfer Matrix...")
try:
    df_cross = pd.read_csv(ROOT_OUT / 'master_tables/master_cross_dataset_results.csv')
    
    if not df_cross.empty:
        train_col = get_col(df_cross, ['train', 'source'])
        test_col = get_col(df_cross, ['test', 'target'])
        f1_col = get_col(df_cross, ['f1', 'f-1'])

        if train_col and test_col and f1_col:
            # Create 4x4 matrix mapping Train -> Test
            t6_matrix = df_cross.pivot_table(index=train_col, columns=test_col, values=f1_col, aggfunc='mean')
            t6_matrix = t6_matrix.round(4)
            
            # Formatting for publication
            t6_matrix.index.name = 'Train Dataset ↓ / Test Dataset →'
            t6_matrix_reset = t6_matrix.reset_index()
            
            t6_matrix_reset.to_csv(ROOT_OUT / 'paper_tables/Table_6_Transfer_Matrix.csv', index=False)
            render_table_to_png(t6_matrix_reset, 'Table_6_Transfer_Matrix', 'Table 6: Cross-Dataset F1 Transfer Matrix')
            print("   ✅ Table 6 saved to paper_tables/")
        else:
            print(f"   ❌ Missing columns in cross-dataset CSV. Found: {df_cross.columns.tolist()}")
    else:
        print("   ❌ Cross-Dataset data is empty.")
except FileNotFoundError:
    print("   ❌ master_cross_dataset_results.csv not found.")

📊 CELL 4: GENERATING TABLES 5 & 6 (ROBUSTNESS & TRANSFER)

📝 Generating Table 5: Robustness Degradation...
   ✅ Table 5 saved to paper_tables/

📝 Generating Table 6: Cross-Dataset Transfer Matrix...
   ✅ Table 6 saved to paper_tables/


In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

print("="*60)
print("📊 CELL 5: GENERATING TABLES 7, 8 & 9 (RUL, DEPLOYMENT, ML vs DL)")
print("="*60)

ROOT_OUT = Path('/kaggle/working/NB-8-AGAIN-outputs')

def get_col(df, keywords):
    """Safely finds a column name prioritizing exact matches, then partial matches."""
    for c in df.columns:
        if c.lower() in [k.lower() for k in keywords]: return c
    for c in df.columns:
        if any(k.lower() in c.lower() for k in keywords): return c
    return None

def render_table_to_png(df, filename, title):
    """Fallback renderer in case Cell 2 is out of memory."""
    fig, ax = plt.subplots(figsize=(max(8, len(df.columns)*1.8), len(df)*0.5 + 1.2))
    ax.axis('tight'); ax.axis('off')
    table = ax.table(cellText=df.values, colLabels=df.columns, loc='center', cellLoc='center')
    table.auto_set_font_size(False); table.set_fontsize(11); table.scale(1.2, 1.8)
    for (row, col), cell in table.get_celld().items():
        if row == 0:
            cell.set_text_props(weight='bold', color='white')
            cell.set_facecolor('#2b3e50')
        elif row % 2 == 0:
            cell.set_facecolor('#f4f4f4')
    plt.title(title, weight='bold', size=14, pad=20)
    plt.savefig(ROOT_OUT / f'paper_tables/{filename}.png', bbox_inches='tight', dpi=300)
    plt.close()

# ---------------------------------------------------------
# TABLE 7: RUL Regression
# ---------------------------------------------------------
print("\n📝 Generating Table 7: RUL Regression Performance...")
try:
    df_rul = pd.read_csv(ROOT_OUT / 'master_tables/master_rul_results.csv')
    if not df_rul.empty:
        model_col = get_col(df_rul, ['model', 'algo'])
        dataset_col = get_col(df_rul, ['dataset', 'data'])
        rmse_col = get_col(df_rul, ['rmse'])
        
        if model_col and rmse_col and dataset_col:
            t7_agg = df_rul.groupby([model_col, dataset_col]).agg({rmse_col: 'mean'}).reset_index()
            t7_pivot = t7_agg.pivot(index=model_col, columns=dataset_col, values=rmse_col).round(3)
            t7_pivot.columns = [f"{col} (RMSE)" for col in t7_pivot.columns]
            t7_pivot.reset_index(inplace=True)
            t7_pivot.rename(columns={model_col: 'Regression Model'}, inplace=True)
            
            t7_pivot.to_csv(ROOT_OUT / 'paper_tables/Table_7_RUL_Regression.csv', index=False)
            render_table_to_png(t7_pivot, 'Table_7_RUL_Regression', 'Table 7: RUL Regression Performance (RMSE)')
            print("   ✅ Table 7 saved to paper_tables/")
        else:
            print(f"   ❌ Missing RUL columns. Found: {df_rul.columns.tolist()}")
    else:
        print("   ❌ RUL data is empty.")
except FileNotFoundError:
    print("   ❌ master_rul_results.csv not found.")

# ---------------------------------------------------------
# TABLE 8: Deployment Metrics
# ---------------------------------------------------------
print("\n📝 Generating Table 8: Deployment & Hardware Metrics...")
try:
    df_dep = pd.read_csv(ROOT_OUT / 'master_tables/master_deployment_metrics.csv')
    if not df_dep.empty:
        model_col = get_col(df_dep, ['model', 'algo'])
        latency_col = get_col(df_dep, ['latency_ms', 'inference_time', 'latency'])
        size_col = get_col(df_dep, ['model_size_mb', 'size', 'memory'])
        
        if model_col:
            agg_dict = {}
            if latency_col: agg_dict[latency_col] = 'mean'
            if size_col: agg_dict[size_col] = 'max'
            
            if agg_dict:
                t8_agg = df_dep.groupby(model_col).agg(agg_dict).reset_index()
                # Rename columns for presentation
                rename_map = {model_col: 'Model'}
                if latency_col: rename_map[latency_col] = 'Avg Inference Latency'
                if size_col: rename_map[size_col] = 'Max Model Size'
                t8_agg.rename(columns=rename_map, inplace=True)
                
                # Round numeric columns safely
                numeric_cols = t8_agg.select_dtypes(include=[np.number]).columns
                t8_agg[numeric_cols] = t8_agg[numeric_cols].round(4)
                
                t8_agg.to_csv(ROOT_OUT / 'paper_tables/Table_8_Deployment_Metrics.csv', index=False)
                render_table_to_png(t8_agg.head(15), 'Table_8_Deployment_Metrics', 'Table 8: Edge Deployment Hardware Metrics')
                print("   ✅ Table 8 saved to paper_tables/")
            else:
                print(f"   ❌ No numeric metrics (latency/size) found. Columns: {df_dep.columns.tolist()}")
        else:
            print("   ❌ Could not find 'model' column in deployment metrics.")
except FileNotFoundError:
    print("   ❌ master_deployment_metrics.csv not found.")

# ---------------------------------------------------------
# TABLE 9: ML vs DL Comparison
# ---------------------------------------------------------
print("\n📝 Generating Table 9: ML vs DL Ultimate Comparison...")
try:
    df_ml = pd.read_csv(ROOT_OUT / 'master_tables/master_classification_results.csv')
    df_dl = pd.read_csv(ROOT_OUT / 'master_tables/master_dl_results.csv')
    
    ml_model_col = get_col(df_ml, ['model', 'algo'])
    ml_f1_col = get_col(df_ml, ['f1', 'f-1'])
    dl_model_col = get_col(df_dl, ['model', 'algo', 'architecture', 'net'])
    dl_f1_col = get_col(df_dl, ['f1', 'f-1'])

    if ml_model_col and ml_f1_col and dl_model_col and dl_f1_col:
        # Aggregate ML
        t9_ml = df_ml.groupby(ml_model_col)[ml_f1_col].mean().reset_index()
        t9_ml.columns = ['Model', 'Global Mean F1']
        t9_ml['Architecture Family'] = 'Classical ML'
        
        # Aggregate DL
        t9_dl = df_dl.groupby(dl_model_col)[dl_f1_col].mean().reset_index()
        t9_dl.columns = ['Model', 'Global Mean F1']
        t9_dl['Architecture Family'] = 'Deep Learning'
        
        combined = pd.concat([t9_ml, t9_dl], ignore_index=True)
        
        # Get the absolute best model per family
        t9_best = combined.loc[combined.groupby('Architecture Family')['Global Mean F1'].idxmax()].copy()
        t9_best['Global Mean F1'] = t9_best['Global Mean F1'].round(4)
        
        # Rearrange columns
        t9_best = t9_best[['Architecture Family', 'Model', 'Global Mean F1']]
        t9_best.rename(columns={'Model': 'Top Performing Model Variant'}, inplace=True)
        
        t9_best.to_csv(ROOT_OUT / 'paper_tables/Table_9_ML_vs_DL.csv', index=False)
        render_table_to_png(t9_best, 'Table_9_ML_vs_DL', 'Table 9: Best ML vs DL Architecture Showdown')
        print("   ✅ Table 9 saved to paper_tables/")
    else:
        print("   ❌ Missing required F1/Model columns to execute ML vs DL showdown.")
except Exception as e:
    print(f"   ❌ Table 9 Error: {e}")

print("\n" + "="*60)
print("🎯 ALL PUBLICATION TABLES (1-9) ARE SUCCESSFULLY COMPLETED.")
print("="*60)

📊 CELL 5: GENERATING TABLES 7, 8 & 9 (RUL, DEPLOYMENT, ML vs DL)

📝 Generating Table 7: RUL Regression Performance...
   ✅ Table 7 saved to paper_tables/

📝 Generating Table 8: Deployment & Hardware Metrics...
   ✅ Table 8 saved to paper_tables/

📝 Generating Table 9: ML vs DL Ultimate Comparison...
   ✅ Table 9 saved to paper_tables/

🎯 ALL PUBLICATION TABLES (1-9) ARE SUCCESSFULLY COMPLETED.


In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as stats
from pathlib import Path
import warnings

warnings.filterwarnings('ignore')

print("="*60)
print("🔬 CELL 6: NON-PARAMETRIC STATISTICAL TESTS & RANKINGS")
print("="*60)

ROOT_OUT = Path('/kaggle/working/NB-8-AGAIN-outputs')
STAT_OUT = ROOT_OUT / 'statistical_tests'

def get_col(df, keywords):
    for c in df.columns:
        if c.lower() in [k.lower() for k in keywords]: return c
    for c in df.columns:
        if any(k.lower() in c.lower() for k in keywords): return c
    return None

def extract_base_model(name):
    """Normalizes model names (e.g., 'lightgbm_seed42' -> 'LightGBM')."""
    name = str(name).lower()
    if 'lightgbm' in name: return 'LightGBM'
    if 'xgboost' in name: return 'XGBoost'
    if 'catboost' in name: return 'CatBoost'
    if 'random_forest' in name: return 'Random Forest'
    if 'logistic' in name: return 'Logistic Regression'
    if 'svm' in name or 'svc' in name: return 'SVM'
    if any(k in name for k in ['lstm', 'gru', 'tcn', 'transformer', 'dl']): return 'Deep Learning'
    return 'Other'

def cohens_d(x, y):
    """Calculates Cohen's d for effect size."""
    nx = len(x); ny = len(y)
    dof = nx + ny - 2
    pooled_std = np.sqrt(((nx-1)*np.std(x, ddof=1)**2 + (ny-1)*np.std(y, ddof=1)**2) / dof)
    return (np.mean(x) - np.mean(y)) / pooled_std if pooled_std > 0 else 0

# ---------------------------------------------------------
# 1. DATA PREPARATION FOR PAIRED TESTS
# ---------------------------------------------------------
print("\n📝 Prepping data for repeated measures...")
try:
    df_class = pd.read_csv(ROOT_OUT / 'master_tables/master_classification_results.csv')
    df_dl = pd.read_csv(ROOT_OUT / 'master_tables/master_dl_results.csv')
    
    # Standardize columns
    c_mod = get_col(df_class, ['model', 'algo']); c_ds = get_col(df_class, ['dataset'])
    c_f1 = get_col(df_class, ['f1', 'f-1']); c_seed = get_col(df_class, ['seed'])
    
    d_mod = get_col(df_dl, ['model', 'algo']); d_ds = get_col(df_dl, ['dataset'])
    d_f1 = get_col(df_dl, ['f1', 'f-1']); d_seed = get_col(df_dl, ['seed'])
    
    # Extract standardized F1 results
    ml_subset = df_class[[c_mod, c_ds, c_seed, c_f1]].rename(columns={c_mod:'Model', c_ds:'Dataset', c_seed:'Seed', c_f1:'F1'})
    dl_subset = df_dl[[d_mod, d_ds, d_seed, d_f1]].rename(columns={d_mod:'Model', d_ds:'Dataset', d_seed:'Seed', d_f1:'F1'}) if not df_dl.empty else pd.DataFrame()
    
    df_stats = pd.concat([ml_subset, dl_subset], ignore_index=True).dropna(subset=['F1'])
    df_stats['Base_Model'] = df_stats['Model'].apply(extract_base_model)
    df_stats['Task_ID'] = df_stats['Dataset'].astype(str) + "_Seed" + df_stats['Seed'].astype(str)
    
    # Pivot to get paired samples (Rows = Tasks, Cols = Base Models, Values = F1)
    # We take the mean in case there are duplicates
    pivot_f1 = df_stats.pivot_table(index='Task_ID', columns='Base_Model', values='F1', aggfunc='mean').dropna()
    
    models = pivot_f1.columns.tolist()
    print(f"   ✅ Created paired sample matrix for models: {models}")
    print(f"   ✅ Total valid paired evaluation tasks (Dataset+Seed pairs): {len(pivot_f1)}")

    # ---------------------------------------------------------
    # 2. FRIEDMAN TEST (GLOBAL SIGNIFICANCE)
    # ---------------------------------------------------------
    stat_report = []
    stat_report.append("=====================================================")
    stat_report.append("          NON-PARAMETRIC STATISTICAL TESTS           ")
    stat_report.append("=====================================================\n")
    
    if len(models) >= 3:
        data_arrays = [pivot_f1[m].values for m in models]
        stat, p_val = stats.friedmanchisquare(*data_arrays)
        stat_report.append("1. FRIEDMAN TEST (Global Differences)")
        stat_report.append(f"   - Null Hypothesis: All models perform equally.")
        stat_report.append(f"   - Chi-Square Statistic: {stat:.4f}")
        stat_report.append(f"   - P-Value: {p_val:.4e}")
        stat_report.append(f"   - Conclusion: {'Reject Null (Significant differences exist)' if p_val < 0.05 else 'Fail to Reject Null (No significant difference)'}\n")
    else:
        stat_report.append("1. FRIEDMAN TEST: Skipped (Requires >= 3 models).\n")

    # ---------------------------------------------------------
    # 3. PAIRWISE WILCOXON SIGNED-RANK & COHEN'S D
    # ---------------------------------------------------------
    stat_report.append("2. PAIRWISE WILCOXON SIGNED-RANK TESTS & EFFECT SIZE")
    
    pairs_to_test = [
        ('LightGBM', 'XGBoost'),
        ('LightGBM', 'CatBoost'),
        ('LightGBM', 'Deep Learning'),
        ('XGBoost', 'Deep Learning'),
        ('Random Forest', 'LightGBM')
    ]
    
    pair_results = []
    for m1, m2 in pairs_to_test:
        if m1 in pivot_f1.columns and m2 in pivot_f1.columns:
            w_stat, w_p = stats.wilcoxon(pivot_f1[m1], pivot_f1[m2])
            d_val = cohens_d(pivot_f1[m1], pivot_f1[m2])
            
            winner = m1 if pivot_f1[m1].mean() > pivot_f1[m2].mean() else m2
            significance = "***" if w_p < 0.001 else "**" if w_p < 0.01 else "*" if w_p < 0.05 else "ns"
            
            pair_results.append({'Model A': m1, 'Model B': m2, 'Winner': winner, 'P-Value': w_p, 'Significance': significance, "Cohen's d": d_val})
            stat_report.append(f"   {m1} vs {m2}:")
            stat_report.append(f"      - P-Value: {w_p:.4e} ({significance})")
            stat_report.append(f"      - Cohen's d: {d_val:.4f} (Effect Size)")
            stat_report.append(f"      - Superior Model: {winner}\n")
            
    df_pairs = pd.DataFrame(pair_results)
    if not df_pairs.empty:
        df_pairs.to_csv(STAT_OUT / 'wilcoxon_pairwise_results.csv', index=False)
        
    with open(STAT_OUT / 'comprehensive_statistical_report.txt', 'w') as f:
        f.write("\n".join(stat_report))
    print("   ✅ Statistical metrics computed and saved.")

    # ---------------------------------------------------------
    # 4. AVERAGE RANK & CD DIAGRAM APPROXIMATION
    # ---------------------------------------------------------
    print("\n📝 Generating Average Rank Visualization...")
    # Calculate ranks (1 is best, so we use descending order for rank calculation)
    ranks = pivot_f1.rank(axis=1, ascending=False)
    avg_ranks = ranks.mean().sort_values()
    
    df_ranks = avg_ranks.reset_index()
    df_ranks.columns = ['Model', 'Average Rank']
    df_ranks.to_csv(STAT_OUT / 'model_average_ranks.csv', index=False)

    # Plotting Average Ranks
    fig, ax = plt.subplots(figsize=(10, len(models)*0.6 + 2))
    y_pos = np.arange(len(avg_ranks))
    
    bars = ax.barh(y_pos, avg_ranks.values, align='center', color='#3498db', edgecolor='black')
    ax.set_yticks(y_pos)
    ax.set_yticklabels(avg_ranks.index, weight='bold')
    ax.invert_yaxis()  # Best rank (lowest number) at the top
    ax.set_xlabel('Average Rank (Lower is Better)', weight='bold')
    ax.set_title('Model Average Rankings Across All Tasks\n(Evaluated via F1 Score)', weight='bold', size=14, pad=15)
    
    # Add rank labels to bars
    for bar in bars:
        width = bar.get_width()
        ax.text(width + 0.05, bar.get_y() + bar.get_height()/2, f'{width:.2f}', 
                ha='left', va='center', weight='bold')
        
    # Formatting
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(axis='x', linestyle='--', alpha=0.7)
    
    plt.savefig(STAT_OUT / 'average_rank_plot.png', bbox_inches='tight', dpi=300)
    plt.close()
    print("   ✅ Average Rank diagram rendered and saved.")

except Exception as e:
    print(f"   ❌ Error during statistical testing: {e}")

print("\n" + "="*60)
print("🎯 STATISTICAL VALIDATION IS COMPLETE.")
print("="*60)

🔬 CELL 6: NON-PARAMETRIC STATISTICAL TESTS & RANKINGS

📝 Prepping data for repeated measures...
   ✅ Created paired sample matrix for models: ['CatBoost', 'Deep Learning', 'LightGBM', 'Logistic Regression', 'Other', 'Random Forest', 'SVM', 'XGBoost']
   ✅ Total valid paired evaluation tasks (Dataset+Seed pairs): 4
   ✅ Statistical metrics computed and saved.

📝 Generating Average Rank Visualization...
   ✅ Average Rank diagram rendered and saved.

🎯 STATISTICAL VALIDATION IS COMPLETE.


In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

print("="*60)
print("🎨 CELL 7: GENERATING PUBLICATION FIGURES (PNG, PDF, SVG)")
print("="*60)

ROOT_OUT = Path('/kaggle/working/NB-8-AGAIN-outputs')
FIG_OUT = ROOT_OUT / 'figures'
FIG_OUT.mkdir(parents=True, exist_ok=True)

# Helper for Safe Column Discovery
def get_col(df, keywords):
    for c in df.columns:
        if c.lower() in [k.lower() for k in keywords]: return c
    for c in df.columns:
        if any(k.lower() in c.lower() for k in keywords): return c
    return None

def save_fig(name):
    """Saves the current matplotlib figure in standard publication formats."""
    for ext in ['png', 'pdf', 'svg']:
        plt.savefig(FIG_OUT / f"{name}.{ext}", bbox_inches='tight', dpi=300, transparent=False, facecolor='white')
    plt.close()

# Set global academic plotting style
plt.rcParams.update({'font.size': 12, 'axes.labelsize': 14, 'axes.titlesize': 16, 'legend.fontsize': 12})
sns.set_theme(style="whitegrid", context="paper")

# ---------------------------------------------------------
# FIG 1: Robustness Degradation Curves
# ---------------------------------------------------------
print("\n📈 Plotting Robustness Degradation Curves...")
try:
    df_rob = pd.read_csv(ROOT_OUT / 'master_tables/master_robustness_results.csv')
    if not df_rob.empty:
        c_mod = get_col(df_rob, ['model', 'algo'])
        c_pert = get_col(df_rob, ['perturbation_type', 'perturb', 'fault'])
        c_int = get_col(df_rob, ['perturbation_level', 'intensity', 'noise'])
        c_f1 = get_col(df_rob, ['f1', 'f-1'])
        
        if all([c_mod, c_pert, c_int, c_f1]):
            # Plot one figure per perturbation type
            for pert in df_rob[c_pert].unique():
                plt.figure(figsize=(10, 6))
                subset = df_rob[df_rob[c_pert] == pert]
                
                # Plot each model's degradation line
                sns.lineplot(data=subset, x=c_int, y=c_f1, hue=c_mod, marker='o', linewidth=2, markersize=8)
                
                plt.title(f"Performance Degradation under {str(pert).replace('_', ' ').title()}", weight='bold')
                plt.xlabel("Perturbation Intensity / Level (%)", weight='bold')
                plt.ylabel("F1 Score", weight='bold')
                plt.legend(title="Model", bbox_to_anchor=(1.05, 1), loc='upper left')
                
                save_fig(f"robustness_curve_{pert}")
            print("   ✅ Robustness curves saved.")
        else:
            print("   ❌ Missing columns for Robustness plot.")
except Exception as e:
    print(f"   ❌ Failed to plot Robustness: {e}")

# ---------------------------------------------------------
# FIG 2: Cross-Dataset Transferability Heatmap
# ---------------------------------------------------------
print("\n🔥 Plotting Transferability Heatmap...")
try:
    df_cross = pd.read_csv(ROOT_OUT / 'master_tables/master_cross_dataset_results.csv')
    if not df_cross.empty:
        c_train = get_col(df_cross, ['train', 'source'])
        c_test = get_col(df_cross, ['test', 'target'])
        c_f1 = get_col(df_cross, ['f1', 'f-1'])
        
        if all([c_train, c_test, c_f1]):
            heatmap_data = df_cross.pivot_table(index=c_train, columns=c_test, values=c_f1, aggfunc='mean')
            
            plt.figure(figsize=(8, 6))
            sns.heatmap(heatmap_data, annot=True, fmt=".3f", cmap="YlGnBu", cbar_kws={'label': 'F1 Score'}, linewidths=1, linecolor='white')
            plt.title("Cross-Dataset Generalization Matrix (F1 Score)", weight='bold', pad=15)
            plt.ylabel("Training Dataset (Source)", weight='bold')
            plt.xlabel("Testing Dataset (Target)", weight='bold')
            
            save_fig("cross_dataset_heatmap")
            print("   ✅ Transfer heatmap saved.")
        else:
            print("   ❌ Missing columns for Heatmap.")
except Exception as e:
    print(f"   ❌ Failed to plot Heatmap: {e}")

# ---------------------------------------------------------
# FIG 3: Deployment Metrics Comparison (Latency vs Memory)
# ---------------------------------------------------------
print("\n⚡ Plotting Deployment Hardware Comparisons...")
try:
    df_dep = pd.read_csv(ROOT_OUT / 'master_tables/master_deployment_metrics.csv')
    if not df_dep.empty:
        c_mod = get_col(df_dep, ['model', 'algo'])
        c_lat = get_col(df_dep, ['latency', 'time'])
        c_mem = get_col(df_dep, ['size', 'memory', 'mb'])
        
        if c_mod and c_lat and c_mem:
            agg_dep = df_dep.groupby(c_mod).agg({c_lat: 'mean', c_mem: 'max'}).reset_index()
            
            # Subplot for Latency and Memory
            fig, axes = plt.subplots(1, 2, figsize=(16, 6))
            
            # Plot 1: Latency
            sns.barplot(data=agg_dep.sort_values(c_lat), y=c_mod, x=c_lat, ax=axes[0], palette="Reds_r")
            axes[0].set_title("Inference Latency Comparison", weight='bold')
            axes[0].set_xlabel("Latency", weight='bold')
            axes[0].set_ylabel("")
            
            # Plot 2: Memory/Size
            sns.barplot(data=agg_dep.sort_values(c_mem), y=c_mod, x=c_mem, ax=axes[1], palette="Blues_r")
            axes[1].set_title("Model Memory Footprint", weight='bold')
            axes[1].set_xlabel("Memory / Size (MB)", weight='bold')
            axes[1].set_ylabel("")
            
            plt.tight_layout()
            save_fig("deployment_hardware_comparison")
            print("   ✅ Deployment hardware charts saved.")
        else:
            print("   ❌ Missing columns for Deployment plot.")
except Exception as e:
    print(f"   ❌ Failed to plot Deployment Metrics: {e}")

print("\n" + "="*60)
print("🎯 ALL FIGURES SUCCESSFULLY RENDERED & SAVED.")
print("="*60)

🎨 CELL 7: GENERATING PUBLICATION FIGURES (PNG, PDF, SVG)

📈 Plotting Robustness Degradation Curves...
   ✅ Robustness curves saved.

🔥 Plotting Transferability Heatmap...
   ✅ Transfer heatmap saved.

⚡ Plotting Deployment Hardware Comparisons...
   ✅ Deployment hardware charts saved.

🎯 ALL FIGURES SUCCESSFULLY RENDERED & SAVED.


In [8]:
import os
import shutil
from pathlib import Path

print("="*60)
print("📦 CELL 8: FINAL ARCHIVE GENERATION & SYSTEM VALIDATION")
print("="*60)

ROOT_OUT = Path('/kaggle/working/NB-8-AGAIN-outputs')
ARCHIVE_NAME = '/kaggle/working/NB-8-AGAIN-FINAL-PAPER-ASSETS'

print("\n🔍 Validating generated asset folders...")
required_dirs = ['master_tables', 'figures', 'statistical_tests', 'paper_tables']
all_passed = True

for d in required_dirs:
    dir_path = ROOT_OUT / d
    if dir_path.exists() and any(dir_path.iterdir()):
        print(f"   ✅ {d}/ contains files.")
    else:
        print(f"   ⚠️ WARNING: {d}/ is missing or empty.")
        all_passed = False

print("\n🗜️ Zipping paper assets into a single downloadable archive...")
try:
    # Shutil automatically appends '.zip' to the end of ARCHIVE_NAME
    shutil.make_archive(ARCHIVE_NAME, 'zip', ROOT_OUT)
    
    zip_size = os.path.getsize(f"{ARCHIVE_NAME}.zip") / (1024 * 1024)
    print(f"   ✅ Archive successfully created: NB-8-AGAIN-FINAL-PAPER-ASSETS.zip ({zip_size:.2f} MB)")
    
    if all_passed:
        print("\n" + "⭐"*30)
        print("🎉 THE PROJECT IS OFFICIALLY COMPLETE AND PAPER-READY! 🎉")
        print("⭐"*30)
        print("\nDownload 'NB-8-AGAIN-FINAL-PAPER-ASSETS.zip' from the Kaggle 'Output' sidebar.")
    else:
        print("\n⚠️ Archive created, but some folders were empty. Review warnings above.")
        
except Exception as e:
    print(f"\n❌ Failed to zip directory: {e}")

📦 CELL 8: FINAL ARCHIVE GENERATION & SYSTEM VALIDATION

🔍 Validating generated asset folders...
   ✅ master_tables/ contains files.
   ✅ figures/ contains files.
   ✅ statistical_tests/ contains files.
   ✅ paper_tables/ contains files.

🗜️ Zipping paper assets into a single downloadable archive...
   ✅ Archive successfully created: NB-8-AGAIN-FINAL-PAPER-ASSETS.zip (3.75 MB)

⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐
🎉 THE PROJECT IS OFFICIALLY COMPLETE AND PAPER-READY! 🎉
⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐

Download 'NB-8-AGAIN-FINAL-PAPER-ASSETS.zip' from the Kaggle 'Output' sidebar.
